# **TFM: Detecció d'esdeveniments importants en partits de futbol a partir de les seves narracions**

**Autor:** Martí Mullor Rordíguez

**Institució:** Universitat Oberta de Catalunya  
**Tutor:** Josep Mª Carmona Leyva

**Data:** Juny 2026

---

### **Nom de l'script: 02_SVM_Baseline**

Aquest script entrena el model Baseline clàssic basat en text. Utilitzant TF-IDF i Support Vector Machines (SVM).

Està compost per:

**0. Importacions**

**1. Configuració inicial i càrrega de dades**

1.1 Muntar Google Drive

1.2 Carregar la configuració inicial

1.3 Definició de rutes i extracció de paràmetres i dades

**2. Preprocessament de text i vetorització (TF-IDF)**

**3. Entrenament del model SVM**

**4. Avaluació i exportació de resultats**

---


## **0. Importacions**

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import joblib
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report

## **1. Configuració inicial i càrrega de dades**

### **1.1. Muntar Google Drive**

In [ ]:
if not os.path.exists('/content/drive'):
    print("Muntant Google Drive...")
    drive.mount('/content/drive')

### **1.2. Carregar la configuració inicial**

In [ ]:
CONFIG_PATH = "/content/drive/MyDrive/TFM/TFM-Deteccio-Esdeveniments-Futbol/config.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

### **1.3. Definició de rutes i extracció de paràmetres i dades**

In [ ]:
# Configuració inicial
paths = config["paths"]
seed = config["global_settings"]["random_seed"]
svm_params = config["svm_baseline"]

# Lectura dels paràmetres de TF-IDF
tfidf_config = svm_params["tfidf"]
vectorizer = TfidfVectorizer(
    max_features=tfidf_config["max_features"],
    stop_words=tfidf_config["stop_words"],
    lowercase=tfidf_config["lowercase"]
)

# Dades preprocessades
print("Carregant els conjunts de dades preprocessats...")
df_train = pd.read_pickle(os.path.join(paths["processed_data"], "train_dataset.pkl"))
df_val = pd.read_pickle(os.path.join(paths["processed_data"], "val_dataset.pkl"))
df_test = pd.read_pickle(os.path.join(paths["processed_data"], "test_dataset.pkl"))

print(f"Mida Train: {df_train.shape} | Val: {df_val.shape} | Test: {df_test.shape}")

## **2. Preprocessament de text i vectorització**

In [ ]:
# Definició de variables
X_train, y_train = df_train['text'], df_train['label']
X_val, y_val = df_val['text'], df_val['label']
X_test, y_test = df_test['text'], df_test['label']

print(f"Dades carregades correctament. Mida Train: {X_train.shape}")

## **3. Entrenament del model SVM**

In [ ]:
print("Vectoritzant el text amb TF-IDF...")
tfidf_config = svm_params["tfidf"]
vectorizer = TfidfVectorizer(
    max_features=tfidf_config["max_features"],
    stop_words=tfidf_config["stop_words"],
    lowercase=tfidf_config["lowercase"]
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("Entrenant el model SVM...")
model_config = svm_params["model"]
class_weight_param = None if model_config["class_weight"] == "None" else model_config["class_weight"]

svm_model = SVC(
    kernel=model_config["kernel"],
    class_weight=class_weight_param,
    probability=model_config["probability"],
    random_state=seed
)

svm_model.fit(X_train_tfidf, y_train)
print("Model entrenat amb èxit.")

## **4. Avaluació i exportació de resultat**

In [ ]:
print("\n--- Avaluació al conjunt de Validació ---")
y_val_pred = svm_model.predict(X_val_tfidf)
print(classification_report(y_val, y_val_pred))

print("\n--- Avaluació al conjunt de Test ---")
y_test_pred = svm_model.predict(X_test_tfidf)
print(classification_report(y_test, y_test_pred))

In [ ]:
# Guardar les prediccions per a l'script d'avaluació final
df_val_results = df_val.copy()
df_val_results['svm_pred'] = y_val_pred

df_test_results = df_test.copy()
df_test_results['svm_pred'] = y_test_pred

df_val_results.to_pickle(os.path.join(paths["results"], "svm_val_predictions.pkl"))
df_test_results.to_pickle(os.path.join(paths["results"], "svm_test_predictions.pkl"))

joblib.dump(vectorizer, os.path.join(paths["models"], "tfidf_vectorizer.pkl"))
joblib.dump(svm_model, os.path.join(paths["models"], "svm_baseline_model.pkl"))
print("Procés finalitzat i arxius guardats.")